In [0]:
"""
========================================================================
Smart Manufacturing Intelligence Platform (SMIP)

Bronze Layer

Notebook : 02_ingest_transactional_data

Author  : Sumanth Vempalle
Version : 1.2.0
========================================================================
"""

# ============================================================================
# Imports
# ============================================================================

from framework.core.configuration import (
    BRONZE_LAYER,
    TRANSACTIONAL_DATA_PATH,
    TRANSACTIONAL_DATASETS,
)

from framework.core.logger import (
    banner,
    info,
    success,
    error,
    line,
)

from framework.io.ingestion import read_csv
from framework.io.delta import write_delta

from framework.quality.validation import validate_dataframe
from framework.quality.metadata import add_audit_columns

from framework.core.session import spark

# ============================================================================
# Start
# ============================================================================

banner("SMIP Bronze Transactional Data Ingestion")

# ============================================================================
# Verify Source Folder
# ============================================================================

info("Checking transactional data folder...")

display(
    dbutils.fs.ls(TRANSACTIONAL_DATA_PATH)
)

# ============================================================================
# Initialize Execution Metrics
# ============================================================================

results = []

total_rows = 0

successful = 0

failed = 0

# ============================================================================
# Processing Order
# ============================================================================

processing_order = [

    "work_orders",

    "production_executions",

    "serial_numbers",

    "press_operations",

    "operator_logins",

    "material_scans",

    "test_results",

    "packaging",

    "force_curve_points"

]

# ============================================================================
# Ingestion Loop
# ============================================================================

banner("Starting Bronze Transactional Ingestion")

for table_name in processing_order:

    file_name = TRANSACTIONAL_DATASETS[table_name]

    try:

        info(f"Processing {file_name}")

        path = f"{TRANSACTIONAL_DATA_PATH}/{file_name}"

        df = read_csv(path)

        rows = validate_dataframe(df)

        df = add_audit_columns(
            df,
            file_name
        )

        write_delta(
            df,
            f"{BRONZE_LAYER}.{table_name}"
        )

        success(f"{table_name} ({rows:,} rows)")

        successful += 1

        total_rows += rows

        results.append({

            "Dataset": table_name,

            "Rows": rows,

            "Status": "SUCCESS"

        })

    except Exception as ex:

        failed += 1

        error(f"{table_name}")

        error(str(ex))

        results.append({

            "Dataset": table_name,

            "Rows": 0,

            "Status": "FAILED"

        })

        line()

# ============================================================================
# Execution Summary
# ============================================================================

banner("Execution Summary")

summary = spark.createDataFrame(results)

display(summary)

print(f"Datasets Processed : {len(processing_order)}")

print(f"Successful         : {successful}")

print(f"Failed             : {failed}")

print(f"Rows Loaded        : {total_rows:,}")

# ============================================================================
# Verify Bronze Tables
# ============================================================================

banner("Bronze Transactional Tables")

display(

    spark.sql(

        f"""

        SHOW TABLES IN {BRONZE_LAYER}

        """

    )

)

# ============================================================================
# Preview Tables
# ============================================================================

for table in processing_order:

    banner(table)

    display(

        spark.table(

            f"{BRONZE_LAYER}.{table}"

        ).limit(5)

    )

banner("Bronze Transactional Ingestion Completed")

success("All transactional datasets processed.")